# Modelling: predicting the `Rings` of an abalone

**Goal:** a first working model that predicts `Rings` (age ≈ `Rings + 1.5` years) from the physical measurements.

**Approach** (based on the conclusions of the EDA):
1. Load the data and drop the invalid rows with **the same rule as in the EDA**.
2. Split into a train set and a held-out test set.
3. Build a scikit-learn `Pipeline`: preprocessing (scale the numerical features, one-hot encode `Sex`) + regressor.
4. Compare a **linear regression** to a **baseline that always predicts the mean** number of rings of the train set. The model is only useful if it beats this baseline.
5. Track every run (parameters, metrics, fitted pipeline) with **MLflow**.

**Metrics:** RMSE (main metric, in rings), MAE (in rings) and R². Each model is evaluated with 5-fold cross-validation on the train set, then once on the test set.

# Imports

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
from mlflow.models import infer_signature
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.model_selection import KFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option("display.max_columns", 500)

# Same plot style as in the EDA notebook
plt.rcParams.update(
    {
        "figure.dpi": 80,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.edgecolor": "#8a8984",
        "axes.labelcolor": "#52514e",
        "axes.titlesize": 11,
        "axes.titleweight": "bold",
        "axes.grid": True,
        "grid.color": "#e6e5e0",
        "grid.linewidth": 0.8,
        "axes.axisbelow": True,
        "xtick.color": "#52514e",
        "ytick.color": "#52514e",
        "legend.frameon": False,
    }
)
MAIN_COLOR = "#2a78d6"
BASELINE_COLOR = "#8a8984"

# Configuration

In [ ]:
# The notebook is run from the notebooks/ folder
ROOT_DIR = Path.cwd().parent
DATA_PATH = ROOT_DIR / "data" / "abalone.csv"

TARGET = "Rings"
NUM_FEATURES = [
    "Length",
    "Diameter",
    "Height",
    "Whole weight",
    "Shucked weight",
    "Viscera weight",
    "Shell weight",
]
CAT_FEATURES = ["Sex"]

TEST_SIZE = 0.2
N_FOLDS = 5
RANDOM_STATE = 42

MLFLOW_TRACKING_URI = f"sqlite:///{ROOT_DIR / 'mlflow.db'}"
MLFLOW_ARTIFACT_LOCATION = (ROOT_DIR / "mlruns").as_uri()
EXPERIMENT_NAME = "abalone-rings"

# Data

## Load and clean

`drop_invalid_rows` applies exactly the cleaning decision of the EDA: drop the rows with `Height == 0`, `Height > 0.4`, or a sum of parts above 1.5 × `Whole weight`.

In [ ]:
def load_data(path: Path) -> pd.DataFrame:
    """Load the raw abalone dataset."""
    return pd.read_csv(path)


def drop_invalid_rows(df: pd.DataFrame) -> pd.DataFrame:
    """Drop the physically impossible rows found in the EDA."""
    parts_sum = df["Shucked weight"] + df["Viscera weight"] + df["Shell weight"]
    invalid = (
        (df["Height"] == 0)
        | (df["Height"] > 0.4)
        | (parts_sum > 1.5 * df["Whole weight"])
    )
    return df[~invalid].reset_index(drop=True)

In [ ]:
df_raw = load_data(DATA_PATH)
df = drop_invalid_rows(df_raw)
n_dropped = len(df_raw) - len(df)
print(f"Removed {n_dropped} rows -> {len(df)} rows left")

## Train / test split

The test set (20%) is kept aside and only used once at the end to report the final scores. Model comparison is done with cross-validation on the train set.

In [ ]:
X = df[NUM_FEATURES + CAT_FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)
print(f"Train: {len(X_train)} rows | Test: {len(X_test)} rows")
print(f"Mean Rings - train: {y_train.mean():.2f} | test: {y_test.mean():.2f}")

# Model pipeline

Every model is wrapped in the same `Pipeline`, so the preprocessing is fitted on the training folds only (no leakage) and the fitted object can be used directly on raw data:

- `StandardScaler` on the numerical features (puts the coefficients of the linear regression on the same scale),
- `OneHotEncoder` on `Sex` (`handle_unknown="ignore"` so an unexpected category does not crash the prediction).

In [ ]:
def build_preprocessor() -> ColumnTransformer:
    """Scale the numerical features and one-hot encode the categorical ones."""
    return ColumnTransformer(
        [
            ("num", StandardScaler(), NUM_FEATURES),
            (
                "cat",
                OneHotEncoder(handle_unknown="ignore", sparse_output=False),
                CAT_FEATURES,
            ),
        ]
    )


def build_pipeline(model) -> Pipeline:
    """Chain the preprocessing and a regressor."""
    return Pipeline([("preprocessor", build_preprocessor()), ("model", model)])

## Models

- **`baseline_mean`**: always predicts the mean `Rings` of the training data. It ignores the features and gives the score to beat.
- **`linear_regression`**: ordinary least squares on all features. Simple and interpretable, as asked for this first version.

In [ ]:
MODELS = {
    "baseline_mean": DummyRegressor(strategy="mean"),
    "linear_regression": LinearRegression(),
}

## Evaluation

In [ ]:
CV_SCORING = {
    "rmse": "neg_root_mean_squared_error",
    "mae": "neg_mean_absolute_error",
    "r2": "r2",
}


def evaluate(y_true: pd.Series, y_pred: np.ndarray) -> dict[str, float]:
    """Regression metrics, in number of rings for RMSE and MAE."""
    return {
        "rmse": root_mean_squared_error(y_true, y_pred),
        "mae": mean_absolute_error(y_true, y_pred),
        "r2": r2_score(y_true, y_pred),
    }


def cross_validate_pipeline(
    pipeline: Pipeline, X: pd.DataFrame, y: pd.Series
) -> dict[str, float]:
    """Mean K-fold scores. sklearn returns negated errors, so we flip the sign back."""
    cv = KFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    scores = cross_validate(pipeline, X, y, cv=cv, scoring=CV_SCORING)
    return {
        "rmse": -scores["test_rmse"].mean(),
        "mae": -scores["test_mae"].mean(),
        "r2": scores["test_r2"].mean(),
    }

# Training with MLflow tracking

Each model (baseline and linear regression) is one MLflow run in the experiment `abalone-rings`. For every run we log:
- **parameters:** model name, model hyperparameters, split settings and data size,
- **metrics:** cross-validation and test RMSE / MAE / R²,
- **model:** the fitted pipeline (preprocessing + regressor) with its input signature.

The tracking data is stored locally (`mlflow.db` and `mlruns/` at the root of the repository). Both are in `.gitignore` and must not be pushed. To compare the runs in the UI, run from the root of the repository:

```bash
uv run mlflow ui --backend-store-uri sqlite:///mlflow.db
```

In [ ]:
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
if mlflow.get_experiment_by_name(EXPERIMENT_NAME) is None:
    mlflow.create_experiment(
        EXPERIMENT_NAME, artifact_location=MLFLOW_ARTIFACT_LOCATION
    )
mlflow.set_experiment(EXPERIMENT_NAME)

In [ ]:
def train_and_log(
    name: str,
    model,
    X_train: pd.DataFrame,
    y_train: pd.Series,
    X_test: pd.DataFrame,
    y_test: pd.Series,
) -> tuple[Pipeline, dict[str, float]]:
    """Cross-validate, fit on the full train set, score on the test set and log it all to MLflow."""
    pipeline = build_pipeline(model)

    with mlflow.start_run(run_name=name):
        mlflow.log_params(
            {
                "model_name": name,
                "test_size": TEST_SIZE,
                "n_folds": N_FOLDS,
                "random_state": RANDOM_STATE,
                "n_train": len(X_train),
                "n_test": len(X_test),
                "n_dropped_rows": n_dropped,
            }
        )
        mlflow.log_params({f"model__{k}": v for k, v in model.get_params().items()})

        cv_metrics = cross_validate_pipeline(pipeline, X_train, y_train)
        pipeline.fit(X_train, y_train)
        test_metrics = evaluate(y_test, pipeline.predict(X_test))

        metrics = {f"cv_{k}": v for k, v in cv_metrics.items()}
        metrics |= {f"test_{k}": v for k, v in test_metrics.items()}
        mlflow.log_metrics(metrics)

        mlflow.sklearn.log_model(
            pipeline,
            name="model",
            signature=infer_signature(X_train, pipeline.predict(X_train)),
            input_example=X_train.head(5),
        )

    return pipeline, metrics

In [ ]:
fitted_pipelines = {}
results = {}
for name, model in MODELS.items():
    fitted_pipelines[name], results[name] = train_and_log(
        name, model, X_train, y_train, X_test, y_test
    )
    print(
        f"{name:<18} CV RMSE = {results[name]['cv_rmse']:.3f} | "
        f"test RMSE = {results[name]['test_rmse']:.3f}"
    )

# Results

In [ ]:
results_df = pd.DataFrame(results).T.sort_values("cv_rmse")
baseline_rmse = results_df.loc["baseline_mean", "test_rmse"]
results_df["test_rmse_gain_vs_baseline"] = 1 - results_df["test_rmse"] / baseline_rmse
results_df.round(3)

In [ ]:
plot_df = results_df.sort_values("test_rmse", ascending=False)

fig, ax = plt.subplots(figsize=(7.5, 3.2))
colors = [
    BASELINE_COLOR if name == "baseline_mean" else MAIN_COLOR for name in plot_df.index
]
bars = ax.barh(plot_df.index, plot_df["test_rmse"], height=0.6, color=colors)
ax.bar_label(bars, fmt="%.2f", padding=3, fontsize=9, color="#0b0b0b")
ax.set_xlabel("Test RMSE (rings, lower is better)")
ax.set_title("Test RMSE: linear regression vs mean baseline")
ax.grid(axis="y", visible=False)
plt.show()

## Linear regression: errors in detail

The EDA showed that the spread of `Rings` grows with the size of the abalone, so we expect larger errors on old abalones.

In [ ]:
best_name = "linear_regression"
best_pipeline = fitted_pipelines[best_name]
y_pred_best = best_pipeline.predict(X_test)
y_pred_baseline = fitted_pipelines["baseline_mean"].predict(X_test)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(y_test, y_pred_best, s=10, alpha=0.4, color=MAIN_COLOR, linewidths=0)
lim = [0, y_test.max() + 1]
ax.plot(lim, lim, color="#8a8984", linewidth=1, linestyle="--")
ax.annotate(
    "perfect prediction",
    (20, 20),
    xytext=(-95, 6),
    textcoords="offset points",
    fontsize=8,
    color="#52514e",
)
ax.set_xlim(lim)
ax.set_ylim(lim)
ax.set_xlabel("True Rings")
ax.set_ylabel("Predicted Rings")
ax.set_title(f"Predicted vs true Rings on the test set ({best_name})")
plt.show()

In [ ]:
rings_bins = pd.cut(
    y_test, bins=[0, 6, 9, 12, 15, 30], labels=["1-6", "7-9", "10-12", "13-15", "16+"]
)
mae_by_bin = pd.DataFrame(
    {
        "n_test": y_test.groupby(rings_bins, observed=True).size(),
        "mae_baseline": (y_test - y_pred_baseline)
        .abs()
        .groupby(rings_bins, observed=True)
        .mean(),
        f"mae_{best_name}": (y_test - y_pred_best)
        .abs()
        .groupby(rings_bins, observed=True)
        .mean(),
    }
)

fig, ax = plt.subplots(figsize=(8, 3.8))
x = np.arange(len(mae_by_bin))
width = 0.38
ax.bar(
    x - width / 2,
    mae_by_bin["mae_baseline"],
    width=width,
    color=BASELINE_COLOR,
    label="Mean baseline",
)
ax.bar(
    x + width / 2,
    mae_by_bin[f"mae_{best_name}"],
    width=width,
    color=MAIN_COLOR,
    label=best_name,
)
ax.set_xticks(x, mae_by_bin.index)
ax.set_xlabel("True Rings")
ax.set_ylabel("MAE (rings)")
ax.set_title("Test MAE by age group")
ax.legend(loc="upper center")
ax.grid(axis="x", visible=False)
plt.show()

mae_by_bin.round(2)

## Coefficients

The features are scaled, so the coefficients are comparable: change in `Rings` for a 1 standard deviation increase of the feature (for `Sex`, relative to the other categories).

In [ ]:
coefs = pd.Series(
    best_pipeline.named_steps["model"].coef_,
    index=best_pipeline.named_steps["preprocessor"].get_feature_names_out(),
).sort_values()
coefs.round(2).to_frame("coefficient")

# Conclusions

**Baseline**
- Predicting the mean (≈ 9.9 rings) gives a test RMSE of **3.44 rings** (R² ≈ 0). This is the score to beat.

**Linear regression**
- It clearly beats the baseline: test **RMSE 2.25** rings (−35%), **MAE 1.57** rings, **R² 0.57**. The cross-validation scores (RMSE 2.16, R² 0.53) are close to the test scores, so the model does not overfit.
- The two largest coefficients are `Whole weight` (+4.5) and `Shucked weight` (−4.3). They have opposite signs because the features are very correlated: for the same whole weight, an abalone with more meat (so less shell) is younger. Because of this multicollinearity, single coefficients should not be over-interpreted.
- Infants (`Sex_I`) get about 0.7–0.8 fewer rings than adults (`Sex_F`, `Sex_M`), as seen in the EDA.

**Where the model fails**
- The error is about 1 ring for abalones with up to 12 rings, but **about 5 rings for 16+**. The model pulls old abalones towards the mean and rarely predicts more than ~17 rings.
- This was expected from the EDA: few old abalones, a non-linear relation (they keep adding rings without growing much), and a spread of `Rings` that grows with size.

**Next steps**
- Try a regularized model (Ridge), features that capture the non-linearity (log of the weights), or a tree-based model.
- Move `load_data`, `drop_invalid_rows` and `build_pipeline` into the `src/` package, so the training script and the API use the same code as this notebook.